# Pre-processing

In [1]:
import os
import pandas as pd
import numpy as np

# Data directory
def find_data_path(target_file='train.csv'):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        if target_file in filenames:
            return dirname
    return None

DATA_DIR = find_data_path()

if DATA_DIR:
    print(f"Data found in: {DATA_DIR}")
    
    # Load Data
    def load_optimized(filename):
        full_path = os.path.join(DATA_DIR, filename)
        if 'train' in filename or 'test' in filename:
            return pd.read_csv(full_path, dtype={
                'userId': np.int32, 
                'movieId': np.int32, 
                'rating': np.float32
            })
        return pd.read_csv(full_path)

    # Load Core Files
    train = load_optimized('train.csv')
    test = load_optimized('test.csv')
    movies = load_optimized('movies.csv')

    # Data Audit & Cold Start Identification
    train_movies = set(train['movieId'].unique())
    test_movies = set(test['movieId'].unique())
    ghost_movies = test_movies - train_movies

    print(f"\n--- DATA AUDIT ---")
    print(f"Training ratings: {len(train):,}")
    print(f"Test predictions: {len(test):,}")
    print(f"Unique Users:     {train['userId'].nunique():,}")
    print(f"Ghost Movies:     {len(ghost_movies)} (Found in test but not train)")
else:
    print("Critical Error: Could not find training data. Ensure the dataset is attached.")

Data found in: /kaggle/input/movie-recommendation-hackathon-2026

--- DATA AUDIT ---
Training ratings: 10,000,038
Test predictions: 5,000,019
Unique Users:     162,541
Ghost Movies:     5044 (Found in test but not train)


# Genre 

In [2]:
from sklearn.preprocessing import MultiLabelBinarizer

# Clean and split genres into lists
# Use a copy to avoid SettingWithCopy warnings
movies_clean = movies.copy()
movies_clean['genres'] = movies_clean['genres'].fillna('(no genres listed)').str.split('|')

# Binary Encoding (0 or 1 for each genre)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies_clean['genres'])
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=movies_clean['movieId'])

# Create a 'Main Genre' for the simple fallback
# First listed genre as the primary category
movies_clean['main_genre'] = movies_clean['genres'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')

print(f"Genre matrix created with {len(mlb.classes_)} unique categories.")
print(f"Example mapping: Movie {movies_clean.iloc[0]['title']} -> {movies_clean.iloc[0]['main_genre']}")

Genre matrix created with 20 unique categories.
Example mapping: Movie Toy Story (1995) -> Adventure


# Titles

In [3]:
import re

def extract_year(title):
    year = re.search(r'\((\d{4})\)', title)
    if year:
        return int(year.group(1))
    return 0 # Default for movies with no year

movies['year'] = movies['title'].apply(extract_year)

# Audit
print(f"Earliest movie year: {movies[movies['year'] > 0]['year'].min()}")
print(f"Latest movie year: {movies['year'].max()}")

Earliest movie year: 1874
Latest movie year: 2019


# Processing the Genome

In [4]:
# Load Genome Scores

input_path = DATA_DIR
# Use float32 to keep the matrix size manageable
genome_scores = pd.read_csv(os.path.join(input_path, 'genome_scores.csv'), 
                            dtype={'movieId': np.int32, 'tagId': np.int32, 'relevance': np.float32})

# Pivot the table: Movies as rows, Tags as columns
# This creates a "feature vector" for every movie
print("Pivoting genome scores... this may take a minute.")
genome_pivot = genome_scores.pivot(index='movieId', columns='tagId', values='relevance')

# Fill missing values
# If a movie is missing from the genome, fill with 0 (neutral relevance)
genome_pivot = genome_pivot.fillna(0)

print(f"Genome Matrix Shape: {genome_pivot.shape}")
print("Each movie now has", genome_pivot.shape[1], "unique tag features.")

Pivoting genome scores... this may take a minute.
Genome Matrix Shape: (13816, 1128)
Each movie now has 1128 unique tag features.


# Cosine Similarity

In [5]:
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import StandardScaler

# Merge the dataframes
content_features = pd.concat([genre_df, genome_pivot], axis=1).fillna(0)

# Convert all column names to strings
content_features.columns = content_features.columns.astype(str)

print(f"Combined features shape: {content_features.shape}")

# Normalize data
# Use StandardScaler because Tag Relevance and Genre (0/1) have different scales
scaler = StandardScaler()
content_features_scaled = scaler.fit_transform(content_features)

print("Scaling complete. RAM is still stable!")

# 4. Re-run the recommendation test function
def get_recommendations(movie_id, top_n=10):
    try:
        idx = content_features.index.get_loc(movie_id)
        movie_vector = content_features_scaled[idx].reshape(1, -1)
        
        # Calculate similarity (Linear kernel is faster for scaled data)
        sim_scores = linear_kernel(movie_vector, content_features_scaled).flatten()
        
        # Get top indices
        related_indices = sim_scores.argsort()[-(top_n+1):-1][::-1]
        
        return content_features.index[related_indices], sim_scores[related_indices]
    except KeyError:
        return None, None

# Test with Toy Story (ID 1)
similar_ids, scores = get_recommendations(1)
if similar_ids is not None:
    print("\nMovies similar to Toy Story (ID: 1):")
    # Join with movies df to see titles
    results = movies[movies['movieId'].isin(similar_ids)]
    print(results[['title', 'genres']])
else:
    print("\nMovie ID 1 not found in content features.")

Combined features shape: (62423, 1148)
Scaling complete. RAM is still stable!

Movies similar to Toy Story (ID: 1):
                              title  \
2264           Bug's Life, A (1998)   
3021             Toy Story 2 (1999)   
4780          Monsters, Inc. (2001)   
6258            Finding Nemo (2003)   
8246        Incredibles, The (2004)   
11361            Ratatouille (2007)   
12429                 WALL·E (2008)   
13362                     Up (2009)   
13942  Welcome to Dongmakgol (2005)   
14813            Toy Story 3 (2010)   

                                                 genres  
2264                Adventure|Animation|Children|Comedy  
3021        Adventure|Animation|Children|Comedy|Fantasy  
4780        Adventure|Animation|Children|Comedy|Fantasy  
6258                Adventure|Animation|Children|Comedy  
8246         Action|Adventure|Animation|Children|Comedy  
11361                          Animation|Children|Drama  
12429       Adventure|Animation|Children|Romance

# Bias-Corrected SVD

In [6]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

# Calculate User Bias (The 'Average' rating for each user)
user_means = train.groupby('userId')['rating'].mean()
global_mean = train['rating'].mean()

# Center the data (Rating - Mean)
train['rating_centered'] = train['rating'] - train['userId'].map(user_means)

# Create Sparse Matrix
# Map IDs to codes to fit the matrix dimensions (0 to N)
u_codes = train['userId'].astype('category').cat.codes
m_codes = train['movieId'].astype('category').cat.codes
user_item_matrix = csr_matrix((train['rating_centered'], (u_codes, m_codes)))

# Run SVD on Centered Data
svd = TruncatedSVD(n_components=50, random_state=42)
user_features = svd.fit_transform(user_item_matrix)
movie_features = svd.components_.T

print("SVD training complete on centered data.")

SVD training complete on centered data.


# Hybrid Prediction Model

In [7]:
import numpy as np
import pandas as pd

def hybrid_prediction(test_df, user_features, movie_features, train_df, movies_df):
    print("Starting prediction engine...")
    
    # Setup ID Mappings
    user_inv_map = {v: k for k, v in dict(enumerate(train_df['userId'].astype('category').cat.categories)).items()}
    movie_inv_map = {v: k for k, v in dict(enumerate(train_df['movieId'].astype('category').cat.categories)).items()}
    global_mean = train_df['rating'].mean()
    
    # Map Test IDs to SVD codes
    test_df['u_idx'] = test_df['userId'].map(user_inv_map).fillna(-1).astype(int)
    test_df['m_idx'] = test_df['movieId'].map(movie_inv_map).fillna(-1).astype(int)
    
    # Initialize global mean
    preds = np.full(len(test_df), global_mean)
    
    # Collaborative Component (SVD)
    known_mask = (test_df['u_idx'] != -1) & (test_df['m_idx'] != -1)
    if known_mask.any():
        u_indices = test_df.loc[known_mask, 'u_idx'].values
        m_indices = test_df.loc[known_mask, 'm_idx'].values
        u_ids = test_df.loc[known_mask, 'userId']
        
        # Get SCD preference
        svd_contribution = np.sum(user_features[u_indices] * movie_features[m_indices], axis=1)

        user_bias = u_ids.map(user_means).fillna(global_mean).values
        preds[known_mask] = svd_contribution + user_bias
    
    # Content Component
    ghost_mask = (test_df['m_idx'] == -1)
    if ghost_mask.any():
        # Get average rating per genre from train
        # We assume movies_df['genres'] is already a list from previous steps
        train_geo = train_df.merge(movies_df[['movieId', 'genres']], on='movieId')
        train_geo['main_genre'] = train_geo['genres'].apply(lambda x: x[0] if isinstance(x, list) else 'Unknown')
        genre_means = train_geo.groupby('main_genre')['rating'].mean().to_dict()
        
        # Map ghost movies to their genre mean
        test_ghosts = test_df[ghost_mask].merge(movies_df[['movieId', 'genres']], on='movieId', how='left')
        test_ghosts['main_genre'] = test_ghosts['genres'].apply(lambda x: x[0] if isinstance(x, list) else 'Unknown')
        preds[ghost_mask] = test_ghosts['main_genre'].map(genre_means).fillna(global_mean).values

    print("Predictions complete.")
    return np.clip(preds, 0.5, 5.0)

# Execute
test['rating'] = hybrid_prediction(test, user_features, movie_features, train, movies)

# Execute and Export
test['rating'] = hybrid_prediction(test, user_features, movie_features, train, movies)
test['Id'] = test['userId'].astype(str) + '_' + test['movieId'].astype(str)
test[['Id', 'rating']].to_csv('submission.csv', index=False)

print("Submission file updated with Bias-Correction. Ready to re-submit!")

Starting prediction engine...
Predictions complete.
Starting prediction engine...
Predictions complete.
Submission file updated with Bias-Correction. Ready to re-submit!
